# Calibrate a heat-diffusion model

Estimate a positive diffusivity by matching a one-dimensional heat simulation to known observations. You should know NumPy arrays; the notebook explains the numerical objective. Learn how shared problem data, per-trial state, contained failures, and application checkpoints fit an adaptive campaign.

## Before you begin

Run the cells in order in a fresh kernel. Keep the complete repository and bundled tutorial data together. The setup below locates your checkout; in Colab it accepts a complete source ZIP. The notebook imports the adjacent Python implementation, so operators and resources have one canonical source.


In [ ]:
from pathlib import Path
import importlib.util
import io
import subprocess
import sys
import zipfile

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "jayrun").is_dir() and (p / "tutorials").is_dir()), None)
if ROOT is None and IN_COLAB:
    from google.colab import files
    uploaded = files.upload()  # Upload the complete Jayrun project ZIP, not just this notebook.
    if len(uploaded) != 1:
        raise ValueError("upload exactly one project ZIP")
    destination = Path("/content/jayrun_tutorial_source")
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
        if sum(item.file_size for item in archive.infolist()) > 500_000_000:
            raise ValueError("archive exceeds this notebook's extraction limit")
        for item in archive.infolist():
            path = (destination / item.filename).resolve()
            if not path.is_relative_to(destination.resolve()) or (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("unsafe archive path or symbolic link")
        archive.extractall(destination)
    candidates = [destination, *destination.iterdir()]
    ROOT = next((p for p in candidates if (p / "jayrun").is_dir() and (p / "tutorials").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Run Jupyter from the extracted project, or upload the full project ZIP in Colab.")
sys.path.insert(0, str(ROOT))
print("Tutorial source:", ROOT)


## Prepare the environment

The next cell checks the packages this lesson uses. Locally, install missing tutorial packages before continuing. In Colab, it installs only missing non-PyTorch packages. For PyTorch lessons, provide a suitable existing PyTorch build; the cell does not replace it.


In [ ]:
required = {'packaging': 'packaging>=25', 'numpy': 'numpy'}
missing = [requirement for name, requirement in required.items() if importlib.util.find_spec(name) is None]
# Colab provides PyTorch. Never replace an existing CPU/CUDA installation here.
if "torch" in missing:
    raise RuntimeError("Install the PyTorch build appropriate for your hardware before running this lesson.")
if missing:
    if not IN_COLAB:
        raise RuntimeError(f"Install these optional tutorial dependencies first: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
from packaging.version import Version
import packaging
if Version(packaging.__version__) < Version("25"):
    raise RuntimeError("This source archive requires packaging >= 25; update it explicitly before proceeding.")


## 1. Understand the objective

The initial temperature is sin(pi*x) on x in [0, 1], with zero boundary values. Analytic observations at time 0.1 use diffusivity 0.17. For each candidate, the solver advances to that horizon; misfit is mean squared error against those observations. Smaller misfit is better. Numerical discretization means the best fitted value need not equal 0.17 exactly.

The explicit Euler step uses dt <= 0.4*dx^2/diffusivity. Smaller cells or larger diffusivity require shorter time steps. ProblemResource holds read-only grid/observations, while each trial has its own changing SimulationState.


In [ ]:
import asyncio
import inspect
import tempfile
import numpy as np
from pathlib import Path
from jayrun import ArtifactContext, Engine
from jayrun.context import ContextState
from jayrun.settings import ContextSettings
from tutorials import heat_calibration as lesson

problem = lesson.make_problem()
print("Cells / horizon:", len(problem.x), problem.horizon)
print(inspect.getsource(lesson.AdvanceSimulation))


## 2. Advance one trial through graph iterations

Each invocation runs a block of numerical steps and outputs its state again. max_iterations=None allows repetition until the operator requests Stop at the horizon. Compare that graph result with the direct numerical loop. A matching result checks integration; the analytic observations provide a separate numerical reference.


In [ ]:
def run_one_trial(diffusivity):
    graph, state = lesson.build_graph()
    with Engine() as engine:
        run = engine.submit(graph, ArtifactContext({state: lesson.initial_state(problem, diffusivity)}),
                            settings=ContextSettings(max_iterations=None, record_history_limit=4))
        run.wait(timeout=30)
        if run.state is not ContextState.FINISHED:
            raise RuntimeError(str(run.report.data.failure))
        return run.artifact(state).value

trial = await asyncio.to_thread(run_one_trial, 0.17)
reference = lesson.initial_state(problem, 0.17)
while reference.time < problem.horizon - 1e-14:
    lesson.advance(reference, problem)
np.testing.assert_allclose(trial.values, reference.values, atol=1e-12, rtol=0)
print("Time / steps / misfit:", trial.time, trial.steps, trial.misfit)


## 3. Refine a campaign and inspect every outcome

The application starts candidates 0.1, 0.2, 0.3, and -0.1. The negative value deliberately fails. It keeps the best successful value and tries two smaller neighborhoods. Four initial candidates and two later sets of three produce ten outcomes. All failures remain in the manifest; they are not silently dropped.


In [ ]:
manifest = await asyncio.to_thread(lesson.run_demo)
print("round  diffusivity  outcome   misfit")
for row in manifest["runs"]:
    print(row["round"], f'{row["diffusivity"]:.4f}', row["outcome"], row.get("misfit", "unavailable"))
assert len(manifest["runs"]) == 10
assert sum(row["outcome"] == "failed" for row in manifest["runs"]) == 1
initial_best = min(row["misfit"] for row in manifest["runs"]
                   if row["round"] == 0 and row["outcome"] == "finished")
assert manifest["best_misfit"] < initial_best
print("Selected diffusivity:", manifest["best_diffusivity"])


## 4. Continue a partial application checkpoint

Save a state after seven solver steps, load it with pickle disabled, and submit it to a new run. Compare its final field with uninterrupted work. The checkpoint is solver state, while experiment_manifest.json records configuration, outcomes, versions, and source/data hashes. graph_id identifies a declaration, not a complete experiment.


In [ ]:
partial = lesson.initial_state(problem, 0.17)
lesson.advance(partial, problem, block_steps=7)
with tempfile.TemporaryDirectory() as temporary:
    checkpoint = Path(temporary) / "partial_solver.npz"
    lesson.save_checkpoint(partial, checkpoint)
    restored = lesson.load_checkpoint(checkpoint)

def continue_solver():
    graph, state = lesson.build_graph()
    with Engine() as engine:
        run = engine.submit(graph, ArtifactContext({state: restored}),
                            settings=ContextSettings(max_iterations=None))
        run.wait(timeout=30)
        if run.state is not ContextState.FINISHED:
            raise RuntimeError(str(run.report.data.failure))
        return run.artifact(state).value

continued = await asyncio.to_thread(continue_solver)
np.testing.assert_allclose(continued.values, trial.values, atol=1e-12, rtol=0)
print("Checkpoint matches uninterrupted:", manifest["checkpoint_matches_uninterrupted"])
print("Provenance fields:", sorted(key for key in manifest if "sha256" in key))


## Try it yourself

Change the first single-trial candidate to 0.1, then compare its misfit with 0.17. Keep checkpoint/uninterrupted comparisons on the same candidate. For a numerical convergence experiment, use 81 cells in both make_problem and build_graph and compare error against analytic observations.

The CLI --output heat_results retains partial_solver.npz and experiment_manifest.json; this notebook uses temporary files. A research release additionally needs an environment lock, external datasets, solver tolerances, and relevant hardware/backend information. The checks do not establish throughput gains or bitwise reproducibility across platforms.
